In [ ]:
import sys
sys.path.append('..')
from NCA.trainer.NCA_trainer import NCA_Trainer
from Common.dataloader.emoji import load_emoji_sequence
from NCA.trainer.data_augmenter_nca import DataAugmenter
from NCA.model.NCA_model import NCA
from einops import rearrange
import time
import jax
import jax.numpy as np
import optax
import matplotlib.pyplot as plt



## Define all the variables

In [ ]:
CHANNELS = 20           # How many channels to use in the model
TRAINING_STEPS = 1000   # How many steps to train for
DOWNSAMPLE = 4          # How much to downsample the image by
NCA_STEPS = 64          # How many NCA steps between each image in the data sequence
BATCHES = 2

## Create a Neural Cellular Automata model
 - There are a few important parameters:
    - `KERNEL_STR` defines what sort of spatial kernels we use
    - `ACTIVATION` accepts any scalar function, acting as the neural network nonlinearity
    - `PADDING` must be "CIRCULAR", "REFLECT" , "REPLICATE" or "ZEROS" - this controls how to handle the borders of the image
    - `FIRE_RATE` must be between 0 and 1 - it is the probability of updating each pixel at each step

In [ ]:
model = NCA(N_CHANNELS=CHANNELS,
            KERNEL_STR=["ID","GRAD","LAP"],
            ACTIVATION=jax.nn.relu,
            PADDING="CIRCULAR",
            FIRE_RATE=0.5)


## Load Data
 - Here we load an individual image from `demo_data/` , and create an initial condition of one seed pixel
 - `load_emoji_sequence` takes a list of strings like `["file_1","file_2",...]` and returns:
   
    an array of shape `[Batch, Timestep, Channels, Width, Height]`, where:
      - `Batch` is currently 1 - this matters more later if we want to train to multiple images at the same time
      - `Timestep` is the length of the input list
      - `Channels` is typically 3 or 4 for colour channels
      - `Width` and `Height` are for the image size

 -  We also use the `DataAugmenter` class, defined in `NCA.trainer.data_augmenter_nca.py`
    - This has a few useful functions for modifying the data during training to produce better results
    - This also adds extra hidden channels to an image
    - By creating subclasses of `DataAugmenter` we can define what behaviour to apply to data during training
         - In this example we just pad the data with extra zeros around the boundary

In [ ]:
data = load_emoji_sequence(["crab.png","crab.png"],impath_emojis="demo_data/",downsample=DOWNSAMPLE)


# For the initial condition, take a small cropped square from the middle of the target image
initial_condition = np.array(data[:,:1])
print(data.shape)
W = initial_condition.shape[-2]
H = initial_condition.shape[-1]
initial_condition = initial_condition.at[0,0,:,:W//2-2].set(0)
initial_condition = initial_condition.at[0,0,:,W//2+1:].set(0)
initial_condition = initial_condition.at[0,0,:,:,:H//2-2].set(0)
initial_condition = initial_condition.at[0,0,:,:,H//2+1:].set(0)


data = np.concatenate([initial_condition,data],axis=1) # Join initial condition and data along the time axis
print("(Batch, Time, Channels, Width, Height): "+str(data.shape))
plt.imshow(rearrange(data,"() T C W H -> W (T H) C" )[...,:3])
plt.show()


class data_augmenter_subclass(DataAugmenter):
    #Redefine how data is pre-processed before training
    def data_init(self,SHARDING=None):
        data = self.return_saved_data()
        data = self.duplicate_batches(data, BATCHES)
        data = self.pad(data, 10) 		
        self.save_data(data)
        return None
    

## Define the trainer object
- The `NCA_Trainer` takes as input the `model`, the `data` and a reference to the `DataAugmenter` class (or a custom subclass)
    - It also takes a `model_filename` for saving the output
- `NCA_Trainer` also logs a lot of training statistics using tensorboard, instructions to read that are below

In [ ]:
trainer = NCA_Trainer(
    NCA_model=model,
    data = data,
    DATA_AUGMENTER=data_augmenter_subclass,
    model_filename="test_grow_crab",
    #model_filename="logging_devtest_1",
    MODEL_DIRECTORY="models/",
    LOG_DIRECTORY="logs/"
)

## Training
- Run the following code cell first, then follow these instructions to view how the training is progressing

### Evaluating training:
- Click on the `wandb` link that shows when running the training



In [ ]:
schedule = optax.exponential_decay(1e-4, transition_steps=TRAINING_STEPS, decay_rate=0.99)
optimiser = optax.chain(
    optax.scale_by_param_block_norm(),
    optax.nadam(schedule)
)

trainer.train(
    t=NCA_STEPS,
    iters=TRAINING_STEPS,
    CLEAR_CACHE_EVERY=TRAINING_STEPS,
    LOOP_AUTODIFF="lax", # Switch to "checkpointed" if you get OOM errors
    REGULARISER_COEFFS={
        "intermediate_state":1.0,
        "contigous_growth":1.0,
    },
    optimiser=optimiser,
    WARMUP=10,
    wandb_args={"project":"NCA","name":"test_grow_crab_1","group":"demo_scripts"},
    LOG_EVERY=50)

## Loading a trained NCA
- Here we can access the trained model as `trainer.NCA_model`
- We can also load NCA model weights with `model.load("demo/models/test_grow_crab.eqx")`, replacing the filename string as needed
    - Note that we have to define the NCA object first, with the correct number of channels and kernels

In [ ]:
model = trainer.NCA_model

In [ ]:
model = NCA(N_CHANNELS=CHANNELS,
            KERNEL_STR=["ID","GRAD","LAP"],
            ACTIVATION=jax.nn.relu,
            PADDING="CIRCULAR",
            FIRE_RATE=0.5)

model = model.load("models/test_grow_crab.eqx")

## Testing trained NCA models
- We can test it on the initial condition, given from `trainer.DATA_AUGMENTER.return_saved_data()[0][0]`

In [ ]:
x0 = trainer.DATA_AUGMENTER.return_saved_data()[0][0]

#print(x0.shape)
#x0 = x0.at[:,:x0.shape[1]//2-1].set(0)

trajectory = model.run(NCA_STEPS,x0)

trajectory = rearrange(trajectory,"T C W H-> T W H C")
print(f"Trajectory shape: {trajectory.shape}")
plt.figure(figsize=(20,10))
plt.imshow(rearrange(trajectory[::10,...,:3],"T W H C -> W (T H) C"))
plt.xticks([])
plt.yticks([])
plt.tight_layout()
plt.show()

### Running a trained NCA on a damaged initial condition
- We can take the initial 3*3 starting seed and remove part of it, or otherwise damage it, and then see what the NCA does



In [ ]:
model = trainer.NCA_model
x0 = trainer.DATA_AUGMENTER.return_saved_data()[0][0]

#print(x0.shape)
x0 = x0.at[:,:x0.shape[1]//2-1].set(0)
x0 = x0.at[:,:,:x0.shape[2]//2-1].set(0)
plt.imshow(rearrange(x0,"C W H -> W H C" )[...,:3])
plt.show()
trajectory = model.run(NCA_STEPS,x0)

trajectory = rearrange(trajectory,"T C W H-> T W H C")
plt.figure(figsize=(20,10))
plt.imshow(rearrange(trajectory[::10,...,:3],"T W H C -> W (T H) C"))
plt.xticks([])
plt.yticks([])
plt.tight_layout()
plt.show()

### Regenerating with a trained NCA
- By starting at half of the target image, can we regrow the other half?

In [ ]:
model = trainer.NCA_model
x0 = trainer.DATA_AUGMENTER.return_saved_data()[0][1]

#print(x0.shape)
#x0 = x0.at[:,:x0.shape[1]//2-1].set(0)
x0 = x0.at[:,:,:x0.shape[2]//2-1].set(0)
plt.imshow(rearrange(x0,"C W H -> W H C" )[...,:3])
plt.show()
trajectory = model.run(NCA_STEPS,x0)

trajectory = rearrange(trajectory,"T C W H-> T W H C")
plt.figure(figsize=(20,10))
plt.imshow(rearrange(trajectory[::10,...,:3],"T W H C -> W (T H) C"))
plt.xticks([])
plt.yticks([])
plt.tight_layout()
plt.show()